In [61]:
#Nama   : Muhammad Farras Ibnu Budiyanto
#NIM    : 240401010085
#Kelas  : IF404


# =============================================================================
# 1. Eksplorasi data awal
# =============================================================================
import pandas as pd, numpy as np
from scipy.stats.mstats import winsorize

print("\n1. Ekplorasi awal")
df = pd.read_csv('housing_dirty.csv')
print("Info Dataset")
print (df.info())
print("Statistik Deskriptif")
print (df.describe(include='all'))
print ("\nJumlah data ksosng")
print (df.isnull().sum())

# =============================================================================
# 2. Menghapus data duplikat
# =============================================================================
print("\n2. Menghapus data duplikat")
print('Sebelum hapus duplikat:', df.shape)
df.drop_duplicates(inplace=True)
print('Setelah hapus duplikat:', df.shape)
#setelah dilakukannya pengecekan data duplikat tidak ditemukan


# =============================================================================
# 3. Menormalisasi string
# =============================================================================
df['kota']    = df['kota'].str.strip().str.title()
df['kondisi'] = df['kondisi'].str.strip().str.lower()


print("\nHasil perbaikan teks :")
print(df[['kota', 'kondisi']].head())
#Memperbaiki kolom kota dengan str.strip() supaya menghapus Spasi yang tidak beraturan
#Memperbaiki kolom kota dengan .str.title() supaya variabel kota mengikuti huruf pertamanya, contoh Yogyakarta.

# ==============================================================================
# 4, IMPUTASI MISSING VALUS DENGAN MEAN UNTUK NUMERIK DAN MODUS UNTUK KATEGORI
# ==============================================================================

df['luas_m2']    = df['luas_m2'].fillna(df['luas_m2'].median())  #mengisi data yang kosong dengan nilai median dari kolom luas_m2
df['harga_juta'] = df['harga_juta'].fillna(df['harga_juta'].median())  #mengisi data yang kosong dengan nilai median dari kolom harga_juta
df['kamar']      = df['kamar'].fillna(df['kamar'].mode()[0]) #mengisi data yang kosong dengan nilai modus dari kolom kamar

# ==============================================================================
# 5. TANGANI OUTLIER DENGAN IQR UNTUK KOLOM 'harga_juta' dan luas_m2
# ==============================================================================
for col in ['harga_juta', 'luas_m2', 'tahun_bangun']: #ditambahkan kolom tahun_bangun karena terdapat data anomali
    Q1, Q3 = df[col].quantile([0.25, 0.75])
    IQR = Q3 - Q1
    df[col] = df[col].clip(Q1-1.5*IQR, Q3+1.5*IQR)

# ==============================================================================
# 6. MELAKUKAN VALIDASI DATA TERAKHIR
# ==============================================================================
assert df.isnull().sum().sum() == 0, 'Waduh, masih ada data yang bolong/missing!'
assert df.duplicated().sum() == 0, 'Waduh, masih ada data yang duplikat!'

print('\n=== VALIDASI SUKSES 100% ===')
print('Shape akhir dataset bersih:', df.shape)
df.to_csv('housing_clean.csv', index=False)
print('Dataset bersih tersimpan!')


# ==============================================================================
# LANGKAH 11 — UPLOAD DATASET RUMAH BERSIH KE API (POST REQUEST)
# ==============================================================================
import requests
import json

URL_POST = "https://jsonplaceholder.typicode.com/posts"

# 2. Mengubah 5 baris pertama data rumah bersih kita menjadi format JSON string
# (Kita kirim 5 sampel saja agar respons dari server tidak kepanjangan)
data_kirim = df.head(5).to_json(orient="records")

# 3. Mengubahnya kembali menjadi Python list/dictionary untuk dikirim
payload = json.loads(data_kirim)

# 4. Kirim data rumah kita ke API menggunakan metode POST
response = requests.post(URL_POST, json=payload, timeout=10)

# 5. Cek apakah server API sukses menerima data kita
# (Untuk POST sukses, status_code biasanya 201 Created)
if response.status_code == 201:
    print("=== BERHASIL UPLOAD DATA RUMAH KE API ===")
    print(f"Status Code: {response.status_code} (Created)")
    print("\nRespons balik dari Server API (Bukti Data Diterima):")
    print(json.dumps(response.json(), indent=4))
else:
    print(f"Gagal Upload! Status Code: {response.status_code}")



# Berdasarkan latihan dengan data set diatas, dapat disimpulkan beberapa poin penting dalam data preprocessing
#1. Penanganan missing value: data kosong (NaN) dalam kolom luas_m2, harga_juta, dan kamar
#2. Penanganan data outlier: terdapat data yang tidak masuk akal seperti harga rumah dan tahun_bangun 9999 padahal baru tahun 2026. data ini berhasil dijinakkan melalui metode IQR
#3. Validasi akhir: menggunakan assert untuk memastikan data sudah bersih dari data kosong dan data yang terduplikasi sebelum di export.
#4. Integritas Rest API: Pada tahap akhir, data rumah yang telah dibersihkan berhasil diintegrasikan dengan jaringan luar melalui HTTP POST Request ke API JSONPlaceholder dengan format JSON



1. Ekplorasi awal
Info Dataset
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 130 entries, 0 to 129
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   id            130 non-null    int64  
 1   luas_m2       112 non-null    float64
 2   harga_juta    113 non-null    float64
 3   kota          130 non-null    object 
 4   kamar         120 non-null    float64
 5   tahun_bangun  130 non-null    int64  
 6   kondisi       130 non-null    object 
dtypes: float64(3), int64(2), object(2)
memory usage: 7.2+ KB
None
Statistik Deskriptif
                id      luas_m2    harga_juta     kota       kamar  \
count   130.000000   112.000000  1.130000e+02      130  120.000000   
unique         NaN          NaN           NaN       31         NaN   
top            NaN          NaN           NaN  Bandung         NaN   
freq           NaN          NaN           NaN       14         NaN   
mean     65.500000   267.627679  8.8563